In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv")

In [ ]:
sample.tail()

In [ ]:
train.head()

In [ ]:
train.shape

In [ ]:
train.info()

In [ ]:
train.isna().sum()

In [ ]:
num_cols = train.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    train[col]=train[col].fillna(train[col].median())

In [ ]:
cat_cols = train.select_dtypes(include=['object']).columns

for col in cat_cols:
    train[col] = train[col].fillna(train[col].mode()[0])

In [ ]:
train.isna().sum()

In [ ]:
for col in train.select_dtypes(include='object').columns:
    print(col,train[col].nunique())

all these cols have not that much categories so we don't need to drop it.

In [ ]:
student_ids = train['id'].copy()

In [ ]:
student_ids.head()

In [ ]:
train.drop(columns='id',inplace=True)

In [ ]:
train.info()

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = train.select_dtypes(include=["object"]).columns

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    encoders[col] = le

In [ ]:
train.head()

In [ ]:
X = train.drop('health_condition',axis=1)
y = train['health_condition']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_val.shape)

In [ ]:
# from sklearn.ensemble import RandomForestClassifier

# rf = RandomForestClassifier(
#     n_estimators=200,
#     random_state=42,
#     n_jobs=-1
# )

# rf.fit(X_train,y_train)

In [ ]:
# pred = rf.predict(X_val)

In [ ]:
# from sklearn.metrics import balanced_accuracy_score

# score = balanced_accuracy_score(y_val, pred)

# print("Balanced Accuracy:", score)

In [ ]:
# feature_importance = (
#     pd.DataFrame({
#         'feature': X.columns,
#         'importance': rf.feature_importances_
#     })
#     .sort_values('importance', ascending=False)
# )

# feature_importance.tail(10)

## Test 

In [ ]:
test.info()

In [ ]:
test.shape

In [ ]:
test.head()

In [ ]:
test.isna().sum()

In [ ]:
num_cols = train.select_dtypes(include=['int64','float64']).columns
cat_cols = train.select_dtypes(include=['object']).columns

for col in num_cols:
    test[col]=train[col].fillna(train[col].median())

for col in cat_cols:
    test[col] = train[col].fillna(train[col].mode()[0])

In [ ]:
test.isna().sum().sum()

In [ ]:
student_ids = test['id'].copy()

In [ ]:
test.drop(columns='id',inplace=True)

In [ ]:
for col in cat_cols:
    test[col] = encoders[col].transform(test[col].astype(str))

In [ ]:
test.shape

In [ ]:
X_test = test.drop(columns=["health_condition"], errors="ignore")

In [ ]:
test_pred = rf.predict(X_test)

In [ ]:
test.head()

In [ ]:
test_pred[:10]

In [ ]:
submission = pd.DataFrame({
    "id": student_ids,
    "health_condition": test_pred
})

In [ ]:
submission.tail(20)

In [ ]:
submission.isna().sum()

In [ ]:
submission["health_condition"] = submission["health_condition"].astype(int)

mapping = {
    0: "unhealthy",
    1: "at-risk",
    2: "fit"
}

submission["health_condition"] = submission["health_condition"].map(mapping)

In [ ]:
submission.head()

In [ ]:
submission.to_csv("submission.csv", index=False)